# 이광수 문체 StyleModel **v2** — Qwen2.5-7B QLoRA (업그레이드판)

v1(3B) 대비 강화: **7B 모델 · LoRA rank 32 · 3 epoch**. 유료 GPU(L4·A100 등) 권장.

오늘 겪은 오류를 모두 예방한 버전:
- `max_length`(SFTConfig 신버전) 사용 · `bf16`(BFloat16 스케일러 에러 회피·Ampere+에서 빠름)
- 네이티브 모델(Qwen, remote code 충돌 없음) · `return_dict=True` 추론 · `packing=False`(교차오염 방지)

**순서:** 런타임 유형을 산 GPU(L4/A100 등)로 선택 → 셀을 위에서부터 실행.

In [ ]:
# 1) 의존성 설치
!pip -q install -U "transformers>=4.44" "trl>=0.9" "peft>=0.12" "bitsandbytes>=0.43" accelerate datasets
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# 2) GPU 확인 — 산 GPU와 메모리 보기 (배치 크기 조절 참고)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
# 메모리별 권장 per_device_train_batch_size (7B·4bit·seq384 기준):
#   16GB(T4) → 2~4 | 24GB(L4) → 4~8 | 40GB(A100) → 8~16

## 3) pairs.jsonl 업로드
로컬 `new-version-multiagent/data/pairs/pairs.jsonl` (4,620쌍) 업로드.

In [ ]:
from google.colab import files
up = files.upload()  # pairs.jsonl 선택
PAIRS = list(up.keys())[0]
print('업로드:', PAIRS)

In [ ]:
# 4) 데이터 로드 + 채팅 포맷
from datasets import load_dataset
from transformers import AutoTokenizer

BASE = 'Qwen/Qwen2.5-7B-Instruct'        # A100이면 'Qwen/Qwen2.5-14B-Instruct' 도 가능
SYSTEM = ('너는 1930~40년대 소설가 이광수(춘원)다. 입력으로 주어진 평이한 현대 한국어 문장을, '
          '이광수 특유의 근대 국어 문체로 다시 써라. 한자어·격식체 어미·예스러운 어휘를 살리고 '
          '뜻은 그대로 보존하라. 변환한 문장만 출력하라.')

tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def to_text(ex):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': ex['neutral']},
        {'role': 'assistant', 'content': ex['lgs']},
    ]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

ds = load_dataset('json', data_files=PAIRS, split='train')
ds = ds.map(to_text, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=0.03, seed=42)
print(ds)
print('--- 예시 ---'); print(ds['train'][0]['text'][:600])

In [ ]:
# 5) 4비트 모델 로드 (QLoRA, bf16 compute)
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # Ampere+에서 빠르고 안정(스케일러 불필요)
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print('loaded', model.config.model_type)

In [ ]:
# 6) LoRA(r=32) + SFTTrainer (epochs=2)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir='lgs_style_lora_v2',
    num_train_epochs=2,                  # 14B는 3epoch면 과적합(결사/요시찰 강제삽입) → 2로
    per_device_train_batch_size=4,       # 메모리 여유 있으면 8~16으로 (위 2번 셀 참고)
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,                           # fp16 아님: BFloat16 스케일러 에러 회피
    max_length=384,
    dataset_text_field='text',
    packing=False,                       # 교차오염 방지(깨끗한 학습)
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=cfg, peft_config=lora,
    train_dataset=ds['train'], eval_dataset=ds['test'],
)
trainer.train()

In [ ]:
# 7) 말투만 평가 — 일상 문장으로 순수 문체 변환 확인 (return_dict 픽스 적용)
import torch
tests = [
    '나는 아침에 일찍 일어나 밥을 먹고 천천히 길을 걸었다.',
    '그 사람은 약속을 잘 지키고 늘 성실하게 일한다.',
    '비가 와서 우산을 들고 시장에 갔는데 사람이 무척 많았다.',
    '친구와 오랜만에 만나 차를 마시며 옛이야기를 나누었다.',
    '봄이 되니 마당의 나무에 새 잎이 돋고 꽃이 피었다.',
    '한 시대를 지배하던 생각이 저물고 새로운 생각이 그 자리를 차지하려 할 때, 사람들은 늘 개혁을 떠올린다.',
]
model.config.use_cache = True
model.eval()
for t in tests:
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':t}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors='pt', return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=True,
                             temperature=0.4, top_p=0.9)
    gen = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    print('[입력]', t)
    print('[근대체]', gen.strip())
    print('-'*60)

In [ ]:
# 8) 어댑터 저장 + 다운로드
trainer.model.save_pretrained('lgs_style_lora_v2_final')
tok.save_pretrained('lgs_style_lora_v2_final')
!zip -r lgs_style_lora_v2_final.zip lgs_style_lora_v2_final >/dev/null
from google.colab import files
files.download('lgs_style_lora_v2_final.zip')
print('완료: lgs_style_lora_v2_final.zip')